# Notebook 1 — The Interpreter

**COMP11212 Week 6 study series.** Approach: the interpreter is the load-bearing artifact. We're building the executable definition of the While language's small-step semantics in Python. Every later notebook uses the trace printer we build here as its primary teaching tool.

## The frame

Most courses teach operational semantics on paper:

$$\langle S, \sigma\rangle \Rightarrow \langle S', \sigma'\rangle$$

The symbols sit there. You stare at them. Eventually they click — or they don't.

We're going to do something different. We're going to write Python code that **is** the operational semantics — and that prints traces in the formal notation. The Python is the implementation, the formal notation is the rendering. **One artifact, two views of the same thing.**

Why this works: when you run a program through the interpreter and watch `⟨S, σ⟩ ⇒ ⟨S', σ'⟩` get produced step by step, the symbols stop being decoration. They start being a recording of something that just happened.

## What you'll be able to do by the end of this notebook

- Explain (in your own words) what each piece of an interpreter does: parser, AST, evaluator, step function, trace.
- Read a small While program and predict its formal trace.
- Use three different views of the same execution (formal / table / dict).
- Use the `check_state` predict-harness to test yourself.

## Active-mode notice

Throughout these notebooks, you'll see cells labelled **🎯 PREDICT**. **Don't run the next cell yet** — fill in your guess first, then run the check cell. This is non-negotiable. The notebooks won't work as a study tool if you skip predictions; you'll just be reading code.

## Setup

We need the `lark` parsing library. Run the cell below if you haven't installed it yet.

In [ ]:
# Uncomment the next line if you haven't installed lark yet:
# !pip install lark

import lark
print(f'lark version: {lark.__version__}')

In [ ]:
import sys
from pathlib import Path

# Make this work whether jupyter was started from notebooks/ or from the repo root.
for candidate in [Path.cwd(), Path.cwd() / 'notebooks', Path.cwd().parent / 'notebooks']:
    if (candidate / 'while_lang.py').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise ImportError("could not find while_lang.py — start jupyter from the notebooks/ folder, or move this notebook next to it")

from while_lang import (
    parse, run, trace,                     # high-level API
    A, B, step, step_iter,                 # the small-step machinery
    Config, Transition, StepBudgetExceeded,
    check_state, check_steps,              # predict harness
)
print('imported OK')

## The 3 C's of CS, in 60 seconds

Part 2 of COMP11212 is organised around three questions:

1. **Correctness** — does the program do what it should? (this needs a formal way to talk about what programs *do* — we're starting that today.)
2. **Complexity** — how does the cost of running it grow as the input grows? (week 7+)
3. **Computability** — what can be computed at all, in principle? (week 9+)

All three need the same foundation: **a precise, formal definition of "what running a program means."** That's operational semantics. That's this notebook (and the next two).

## What is the While language?

While is a tiny imperative language. It has:

- **Variables** holding integers (any integer — no overflow).
- **Arithmetic expressions**: `+`, `-`, `*`, numerals, variable references.
- **Boolean expressions**: `tt`, `ff`, `=`, `<=`, `!` (not), `&` (and).
- **Statements**: `:=` assignment, `skip` (does nothing), `;` sequential composition, `if-then-else`, `while-do`.

The chapter writes some of these with fancy symbols (`≤`, `¬`, `∧`, `×`). When typing programs into Python we use ASCII (`<=`, `!`, `&`, `*`). The interpreter's *output* uses the formal symbols.

Here's a complete While program — the chapter's Example 1 (integer division):

In [ ]:
division_prog = '''
    r := m;
    d := 0;
    while n <= r do (
        d := d + 1;
        r := r - n
    )
'''

# Run it with m=10, n=3. Final state should give us d=3 (10 div 3) and r=1 (10 mod 3).
result = run(division_prog, {'m': 10, 'n': 3})
print(result)

Good. The final state shows `d=3` and `r=1`, exactly as the chapter predicts (10 = 3·3 + 1, so 10 div 3 = 3 and 10 mod 3 = 1).

Now the question: **what just happened?** What did `run()` actually do, mechanically? That's what the rest of this notebook unpacks.

## The pieces of an interpreter

Five things, in this order:

1. **Parser** — text in, AST out. Turns the string `r := m; d := 0; ...` into Python objects we can work with.
2. **AST** — abstract syntax tree. One Python class per BNF non-terminal alternative.
3. **Evaluators `A` and `B`** — given an expression and a state σ, compute the integer (for `A`) or boolean (for `B`) it denotes.
4. **`step`** — given a configuration `⟨S, σ⟩`, produce one transition `⟨S, σ⟩ ⇒ ⟨S', σ'⟩` by applying exactly one rule of the small-step semantics.
5. **`trace`** — repeatedly call `step` until you hit `skip`, collect transitions, render them in one of three views.

Let's look at each in turn.

## §1. The AST — program text as Python objects

The chapter defines the syntax of While via Backus-Naur grammar. For arithmetic expressions:

```
<aexp> ::= <num> | <var> | <aexp> + <aexp> | <aexp> − <aexp> | <aexp> × <aexp>
```

That's a recursive definition: an arithmetic expression is *either* a numeral, a variable, or built from two smaller arithmetic expressions joined by `+`, `−`, or `×`. 

We translate this directly into Python — one `@dataclass` per alternative:

```python
@dataclass(frozen=True)
class Num:    value: int
@dataclass(frozen=True)
class Var:    name: str
@dataclass(frozen=True)
class Add:    left: AExp; right: AExp
@dataclass(frozen=True)
class Sub:    left: AExp; right: AExp
@dataclass(frozen=True)
class Mul:    left: AExp; right: AExp
```

An `AExp` is *one of* these five things. The recursive cases (`Add`, `Sub`, `Mul`) hold smaller `AExp` objects inside — same as the BNF.

Boolean expressions and statements get the same treatment. See `while_lang.py` for the full set.

Let's parse a program and look at the resulting tree:

In [ ]:
ast = parse('x := 5 + 3 * y')
print(ast)

Read that carefully. It's an `Assign` to variable `x` with the expression `5 + (3 * y)`. Note that:

- Multiplication binds tighter than addition (so `5 + 3 * y` parses as `5 + (3 * y)`, not `(5 + 3) * y`).
- The variable `y` becomes `Var(name='y')` — the parser doesn't know its value, just its name.
- The numeral `5` becomes `Num(value=5)` — an integer wrapped in a class.

This is what we mean by **abstract syntax**. The string `'x := 5 + 3 * y'` is *concrete* — it has parentheses concerns, whitespace, all the messy stuff. The AST is *abstract* — a clean tree where structure is unambiguous.

### 🎯 PREDICT

Before running the next cell, predict: what AST do you get when you parse `'if x = 0 then (y := 1) else (y := 2)'`?

Write your guess below as a Python expression using the AST classes. (You won't be exactly right syntactically — just sketch the shape.)

Hint: it will be an `If(...)` with three children — the condition, the then-branch, the else-branch.

In [ ]:
# YOUR PREDICTION (in plain English or a sketch — no need to be syntactically perfect):
my_prediction = '''
If(
    cond = ...,
    then_branch = ...,
    else_branch = ...,
)
'''
print(my_prediction)

# Now run the next cell to see the actual AST.

In [ ]:
ast = parse('if x = 0 then (y := 1) else (y := 2)')
print(ast)

## §2. The state σ — a function from variables to integers

Definition 1 in chapter 2 says:

> A **state** is a function σ : Vars → ℤ such that there are only finitely many variables `x` with σ(x) ≠ 0.

In Python we model this with a sparse dict: explicit entries for variables we care about, default-zero for everything else.

When we write `σ = {m ↦ 10, n ↦ 3}` in the chapter, that's a state where `m` is 10, `n` is 3, and *every other variable* is 0. In Python:

```python
sigma = {'m': 10, 'n': 3}    # x, y, z, ... all default to 0
```

The interpreter uses `sigma.get(name, 0)` to query a variable — if the name isn't in the dict, you get 0. That's how the "finitely many non-zero" constraint becomes practical.

In [ ]:
sigma = {'m': 10, 'n': 3}

print(f'σ(m) = {sigma.get("m", 0)}')      # 10 — explicit
print(f'σ(n) = {sigma.get("n", 0)}')      # 3 — explicit
print(f'σ(banana) = {sigma.get("banana", 0)}')  # 0 — default

### State updates — `σ[x ↦ n]`

Updating a state is how the program changes things. The chapter writes `σ[x ↦ n]` for "the state σ but with x mapped to n." Mathematically:

$$(\sigma[x \mapsto n])(y) = \begin{cases} n & y = x \\ \sigma(y) & \text{otherwise} \end{cases}$$

In Python that's just `{**sigma, x: n}` — copy the old dict, set the new value:

In [ ]:
sigma = {'m': 10, 'n': 3}
sigma_updated = {**sigma, 'r': 10}        # σ[r ↦ 10]
print(sigma_updated)

# σ itself is unchanged — updates produce a new state, they don't mutate the old one
print(sigma)

## §3. The evaluators `A` and `B`

These are the workhorses of expression evaluation. Both take an expression and a state, both are defined recursively over the structure of the expression — exactly like the chapter's §2.3.1 and §2.3.2.

**`A` for arithmetic** (returns int):

| AST node | Meaning |
|---|---|
| `Num(n)` | `n` |
| `Var(x)` | `σ(x)` (which is 0 if not in σ) |
| `Add(a, b)` | `A(a, σ) + A(b, σ)` |
| `Sub(a, b)` | `A(a, σ) − A(b, σ)` |
| `Mul(a, b)` | `A(a, σ) · A(b, σ)` |

**`B` for boolean** (returns bool):

| AST node | Meaning |
|---|---|
| `BTrue` | `tt` |
| `BFalse` | `ff` |
| `Eq(a, b)` | `tt` if `A(a, σ) = A(b, σ)` else `ff` |
| `Le(a, b)` | `tt` if `A(a, σ) ≤ A(b, σ)` else `ff` |
| `Not(b)` | `tt` if `B(b, σ) = ff` else `ff` |
| `And(b₁, b₂)` | `tt` if both `B(b₁,σ) = tt` and `B(b₂,σ) = tt` else `ff` |

**Important:** these are total functions. Given any state and any expression, they return a value. There's no "undefined" case (no division, no overflow — `A` always returns an integer, `B` always returns a boolean).

In [ ]:
# Build small ASTs by parsing 'x := <expr>' and pulling out the .expr field.
expr_5_plus_3y = parse('x := 5 + 3 * y').expr

print('A(5 + 3*y) in {y: 4} =', A(expr_5_plus_3y, {'y': 4}))    # 5 + 12 = 17
print('A(5 + 3*y) in {} =', A(expr_5_plus_3y, {}))             # 5 + 0 = 5
print('A(5 + 3*y) in {y: -1} =', A(expr_5_plus_3y, {'y': -1})) # 5 + -3 = 2

In [ ]:
# A boolean expression: (x <= 10) & !(x = 0)
bool_expr = parse('if x <= 10 & !(x = 0) then (y := 1) else (y := 2)').cond

print('B in {x: 0} =', B(bool_expr, {'x': 0}))     # 0 ≤ 10 is true, but x = 0 is true, so ¬(x = 0) is false → AND is false
print('B in {x: 5} =', B(bool_expr, {'x': 5}))     # 5 ≤ 10 true, ¬(5 = 0) true → true
print('B in {x: 11} =', B(bool_expr, {'x': 11}))   # 11 ≤ 10 false → AND false

### 🎯 PREDICT

Given the state σ = `{a: 3, b: 7}`, what does `A((a + b) * 2, σ)` evaluate to? Fill in your guess below.

In [ ]:
# YOUR PREDICTION:
my_prediction = ...   # replace with an integer

# CHECK:
expr = parse('z := (a + b) * 2').expr
actual = A(expr, {'a': 3, 'b': 7})
print(f'predicted: {my_prediction},  actual: {actual},  {"✅" if my_prediction == actual else "❌"}')

## §4. `step` — one ⇒ at a time

This is **the** central function. Given a configuration `⟨S, σ⟩`, `step` produces one transition.

There's exactly one rule per kind of statement (and one extra rule for `skip;S` and one general rule for `S;T`). Here they are, side by side with the Python:

### Assignment rule (`:=`)

$$\langle x := a,\ \sigma\rangle \Rightarrow \langle \textbf{skip},\ \sigma[x \mapsto \mathcal{A}\,a\,\sigma]\rangle$$

```python
if isinstance(s, Assign):
    new_sigma = {**sigma, s.var: A(s.expr, sigma)}
    return Transition(cfg, Config(Skip(), new_sigma), ":=")
```

### Sequential composition rule (two cases)

If the left side is `skip`, drop it:
$$\langle \textbf{skip}; T,\ \sigma\rangle \Rightarrow \langle T,\ \sigma\rangle$$

Otherwise, take a step of the left side, keep the right side:
$$\frac{\langle S,\ \sigma\rangle \Rightarrow \langle S',\ \sigma'\rangle}{\langle S; T,\ \sigma\rangle \Rightarrow \langle S'; T,\ \sigma'\rangle}$$

### `if` rule (two cases)

Branch on the boolean condition:
$$\langle \textbf{if } b \textbf{ then } S \textbf{ else }(S'),\ \sigma\rangle \Rightarrow \begin{cases}\langle S,\ \sigma\rangle & \mathcal{B}\,b\,\sigma = \mathbf{tt} \\ \langle S',\ \sigma\rangle & \text{otherwise}\end{cases}$$

### `while` rule (two cases)

Either unfold or skip:
$$\langle \textbf{while } b \textbf{ do }(S),\ \sigma\rangle \Rightarrow \begin{cases}\langle S; \textbf{while } b \textbf{ do }(S),\ \sigma\rangle & \mathcal{B}\,b\,\sigma = \mathbf{tt} \\ \langle \textbf{skip},\ \sigma\rangle & \text{otherwise}\end{cases}$$

**Notice the `while-tt` rule:** it expands into `S; while b do (S)`. The same `while` reappears on the right-hand side. That's why traces can grow arbitrarily — the program text gets *bigger* during a loop iteration before it shrinks. 

Let's see `step` in action on a tiny program:

In [ ]:
# A 2-statement program. We'll call step manually several times.
from while_lang import cfg_to_str

cfg = Config(parse('x := 5; y := x + 1'), {})
print(f'start: {cfg_to_str(cfg)}')

for i in range(1, 10):
    t = step(cfg)
    if t is None:
        print(f'  step {i}: terminated')
        break
    print(f'  step {i}: {cfg_to_str(t.after)}    [{t.rule}]')
    cfg = t.after

Trace this carefully:

1. `x := 5; y := x + 1` — start. Sequential composition, left side is an assignment.
2. After step 1: the `;` rule fires. Inside, the assignment rule reduces `x := 5` to `skip` and updates state. The right side `y := x + 1` is preserved unchanged.
3. After step 2: now we have `skip; y := x + 1`. The `skip-;` rule fires — drop the skip.
4. After step 3: just `y := x + 1`. This is a single assignment, not inside a sequential composition. The assignment rule fires — `y` becomes 6 (because `x` was updated to 5 earlier).
5. After step 4: `skip` reached. The program is terminal. Next call to `step` returns `None`.

Three of the four transitions are about **administrative bookkeeping** (`;` and `skip-;` rules). Only one actually changes state visibly. This is what "small step" means — every micro-operation gets its own transition. Big-step semantics (the appendix; not examinable) collapses all of these into one super-step.

### 🎯 PREDICT

How many transitions does this program take to terminate, starting from `{}`?

```
if x = 0 then (y := 1) else (y := 2)
```

Hint: count the if-rule firing as 1 transition. Then count the assignment rule. The assignment-rule firing leaves `skip` — but is that a transition? (No: `skip` is terminal, no further rule applies.)

In [ ]:
# YOUR PREDICTION:
my_prediction = ...   # replace with an integer

# CHECK:
check_steps(my_prediction, 'if x = 0 then (y := 1) else (y := 2)', {})

## §5. `trace` — three views of the same execution

`trace` repeatedly calls `step` until the program reaches `skip` (or hits a step budget). It then renders the sequence of transitions in one of three views:

1. **`view='formal'`** — Example 7 style. Greek letters, `⇒` arrows, rule names. **This is what the exam asks you to produce by hand.**
2. **`view='table'`** — Example 8 / 11 style. One row per transition, columns for variables that change. Easier to scan when traces get long.
3. **`view='dict'`** — final state only. Returns a Python dict. Useful for asserting correctness, useless for understanding *why*.

All three are the same execution, rendered differently. Here's the chapter's Example 1 in all three:

In [ ]:
print(trace(division_prog, {'m': 10, 'n': 3}, view='formal'))

In [ ]:
print(trace(division_prog, {'m': 10, 'n': 3}, view='table'))

In [ ]:
print(trace(division_prog, {'m': 10, 'n': 3}, view='dict'))

Compare these three. The formal view is the densest in information — every transition labelled with the rule that fired, the program text shown in full each step (with the long while-loop body abbreviated as `L1`, the same convention the chapter uses in Example 7).

The table view is what you'd produce on the exam if asked to "track the execution of this program" — it shows each step's effect on the variables, with the noise of the `;` and `skip-;` rules made visible but compressed.

The dict view is just the answer.

## §6. The predict harness

We've already used `check_steps` once. There's also `check_state` — predict the final state, get told if you're right and shown the trace if you're not.

In [ ]:
# Correct prediction → green tick + final state.
check_state(
    predicted={'m': 10, 'n': 3, 'r': 1, 'd': 3},
    prog=division_prog,
    state={'m': 10, 'n': 3},
)

In [ ]:
# Wrong prediction → red X + diff + the formal trace so you can see where you went wrong.
check_state(
    predicted={'m': 10, 'n': 3, 'r': 0, 'd': 4},   # wrong
    prog=division_prog,
    state={'m': 10, 'n': 3},
)

## What you have now

- `parse(source)` → AST.
- A state σ = a Python dict, with `{**σ, x: n}` as the update operation.
- `A(expr, σ)` and `B(expr, σ)` evaluate expressions.
- `step(cfg)` produces one transition; `step_iter(prog, σ)` is a lazy iterator.
- `trace(prog, σ, view=...)` renders the whole execution.
- `check_state(...)` and `check_steps(...)` for the active-mode predict cells.

**Crucially**: the trace printer's output **is** the formal small-step semantics, exactly as the chapter writes it. When you stare at `⟨S, σ⟩ ⇒ ⟨S', σ'⟩` in the trace output, you are looking at the formal notation — produced by Python you understand. The two are the same artifact.

## Where we go next

- **Notebook 2 (Syntax):** zoom in on the BNF grammars from chapter 1, walk all 6 chapter examples, work exercises 1–10. We use the interpreter's parser to verify our understanding of the grammar.
- **Notebook 3 (Semantics):** zoom in on chapter 2 §2.1–2.4 — states, the rules, examples 7–11. Exercise 14 (the gcd derivation) is the showpiece.
- **Notebook 4 (Reasoning):** non-termination, Propositions 2 & 3, exercises 17–18 (proof-style), the quiz, and the closing "translate Python intuition into exam-style proofs" section.

Open `02_syntax.ipynb` next.